# Database Agents & Autonomous Text-to-SQL Systems

In this notebook, we build an intelligent **SQL Database Agent** using LangChain and SQLAlchemy. The agent dynamically discovers database schemas, constructs valid SQL queries from natural language, executes them safely, and synthesizes human-readable insights.

### Key Concepts Covered:
1. **Zero-Setup Standalone Database**: Embedded SQLite database  created on-the-fly, with optional fallback/switch to MySQL.
2. **Local LLM**: Uses the Ollama model from `config.yaml`.
3. **SQLDatabaseToolkit**: Dynamically querying table names, fetching table schemas, and running verification queries.
4. **ReAct Loop & Query Guardrails**: Limiting result sets (`top_k`), preventing destructive statements (DML/DDL), and error self-correction.

In [ ]:
# Install dependencies if needed
#!pip install -U langchain langchain-community langchain-openai sqlalchemy sqlite3 pyyaml

## 1. Environment & Dual LLM Backend Setup

In [ ]:
import os
import sqlite3
from pprint import pprint
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_classic import hub
from langchain.agents import create_agent
import yaml
from langchain_openai import ChatOpenAI

cfg = yaml.safe_load(open("config.yaml"))

# Ollama serves an OpenAI-compatible API under /v1; settings come from config.yaml
model_name = cfg["ollama"]["langchain_model"]   # larger than the Strands model: see config.yaml
base_url = cfg["ollama"]["url"].rstrip("/") + "/v1"

print(f"Initializing LLM: {model_name} (Base URL: {base_url})")
llm = ChatOpenAI(
    model=model_name,
    temperature=0.0,
    base_url=base_url,
    api_key="ollama",           # ignored by Ollama, required by the client
    max_retries=2,
)


## 2. Setting Up the Relational Database

We create a self-contained SQLite database (path `tools.sales_db` in `config.yaml`) containing realistic enterprise entities:
- `employees`: Staff directory with titles, office locations, and reports-to hierarchy.
- `customers`: Customer accounts, sales representatives, and geographical data.
- `products`: Product catalog, product lines, and inventory pricing.
- `orders`: Transaction headers with order status and timestamps.

In [ ]:
db_file = cfg["tools"]["sales_db"]
conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# Create Tables
cursor.executescript("""
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS employees;

CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    job_title TEXT NOT NULL,
    email TEXT NOT NULL,
    reports_to INTEGER,
    office_location TEXT NOT NULL
);

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    country TEXT NOT NULL,
    city TEXT NOT NULL,
    sales_rep_id INTEGER,
    credit_limit REAL,
    FOREIGN KEY (sales_rep_id) REFERENCES employees(employee_id)
);

CREATE TABLE products (
    product_code TEXT PRIMARY KEY,
    product_name TEXT NOT NULL,
    product_line TEXT NOT NULL,
    buy_price REAL NOT NULL,
    msrp REAL NOT NULL,
    quantity_in_stock INTEGER NOT NULL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    order_date TEXT NOT NULL,
    customer_id INTEGER NOT NULL,
    status TEXT NOT NULL,
    total_amount REAL NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
""")

# Populate Sample Records
employees_data = [
    (1001, "Diane", "Murphy", "President", "dmurphy@classicmodels.com", None, "San Francisco"),
    (1002, "Mary", "Patterson", "VP Sales", "mpatterson@classicmodels.com", 1001, "San Francisco"),
    (1003, "Jeff", "Firrelli", "VP Marketing", "jfirrelli@classicmodels.com", 1001, "San Francisco"),
    (1004, "William", "Patterson", "Sales Manager", "wpatterson@classicmodels.com", 1002, "Sydney"),
    (1005, "Gerard", "Bondur", "Sales Manager", "gbondur@classicmodels.com", 1002, "Paris"),
    (1006, "Anthony", "Bow", "Sales Rep", "abow@classicmodels.com", 1004, "Sydney"),
    (1007, "Leslie", "Jennings", "Sales Rep", "ljennings@classicmodels.com", 1005, "Paris"),
    (1008, "Leslie", "Thompson", "Sales Rep", "lthompson@classicmodels.com", 1005, "Paris")
]
cursor.executemany("INSERT INTO employees VALUES (?,?,?,?,?,?,?)", employees_data)

customers_data = [
    (101, "Atelier graphique", "France", "Nantes", 1007, 21000.0),
    (102, "Signal Gift Stores", "USA", "Las Vegas", 1006, 71800.0),
    (103, "Australian Collectors, Co.", "Australia", "Melbourne", 1006, 117300.0),
    (104, "La Rochelle Gifts", "France", "Nantes", 1007, 118200.0),
    (105, "Baane Mini Imports", "Norway", "Stavern", 1008, 81700.0),
    (106, "Euro+ Shopping Channel", "Spain", "Madrid", 1008, 227600.0),
    (107, "Corrida Auto Replicas", "Spain", "Madrid", 1008, 104600.0),
    (108, "Dragon Souveniers, Ltd.", "Singapore", "Singapore", 1006, 103800.0)
]
cursor.executemany("INSERT INTO customers VALUES (?,?,?,?,?,?)", customers_data)

products_data = [
    ("S10_1678", "1969 Harley Davidson Ultimate Chopper", "Motorcycles", 48.81, 95.70, 7933),
    ("S10_1949", "1952 Alpine Renault 1300", "Classic Cars", 98.58, 214.30, 7305),
    ("S10_2016", "1996 Moto Guzzi 1100i", "Motorcycles", 68.99, 118.94, 6625),
    ("S10_4698", "2003 Harley-Davidson Eagle Drag Bike", "Motorcycles", 91.02, 193.66, 5582),
    ("S12_1099", "1968 Ford Mustang", "Classic Cars", 95.34, 194.57, 6883)
]
cursor.executemany("INSERT INTO products VALUES (?,?,?,?,?,?)", products_data)

orders_data = [
    (10001, "2024-01-10", 106, "Shipped", 58240.50),
    (10002, "2024-01-15", 103, "Shipped", 34120.00),
    (10003, "2024-02-01", 106, "Shipped", 74210.80),
    (10004, "2024-02-18", 101, "Resolved", 12500.00),
    (10005, "2024-03-05", 107, "Shipped", 45000.00),
    (10006, "2024-03-22", 106, "In Process", 62300.00)
]
cursor.executemany("INSERT INTO orders VALUES (?,?,?,?,?)", orders_data)
conn.commit()
conn.close()

print(f"SQLite database successfully generated: {db_file}")

## 3. Initializing LangChain SQL Toolkit

LangChain integrates with SQLAlchemy to inspect table schemas and generate standard SQL queries.

In [ ]:
# Connect via SQLAlchemy URI (defaults to SQLite, or can use MySQL if configured)
mysql_uri = os.getenv("MYSQL_DATABASE_URI", None)
db_uri = mysql_uri if mysql_uri else f"sqlite:///{db_file}"

db = SQLDatabase.from_uri(db_uri)
print(f"Dialect: {db.dialect}")
print(f"Usable Tables: {db.get_usable_table_names()}")

toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()
print("\nAvailable Agent Tools:")
for t in tools:
    print(f" • {t.name}: {t.description[:80]}...")

## 4. Building the Text-to-SQL ReAct Agent

We pull the standard SQL prompt template from LangChain Hub or build a structured system prompt that instructs the LLM on SQL best practices (e.g. casting types, aliasing columns, handling NULLs).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run, then look at the results of the query and return the answer.
Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most {top_k} results using LIMIT clause.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.
You have access to tools for interacting with the database.
Only use the below tools. Only use the information returned by the below tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
""".format(dialect=db.dialect, top_k=5)

try:
    from langchain.agents import create_agent
    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=system_prompt
    )
except Exception:
    from langchain_community.agent_toolkits import create_sql_agent
    agent = create_sql_agent(
        llm=llm,
        db=db,
        verbose=True,
        handle_parsing_errors=True,
        max_iterations=6
    )

## 5. Executing Natural Language Database Queries

In [ ]:
def ask_database(question: str):
    print(f"\nQuery: {question}")
    print("=" * 60)
    try:
        if hasattr(agent, "invoke"):
            res = agent.invoke({"input": question} if "input" in getattr(agent, "input_keys", ["input"]) else question)
            ans = res.get("output", res) if isinstance(res, dict) else res
        else:
            ans = agent.run(question)
        print(f"Response:\n{ans}")
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
# Query 1: Counting records
ask_database("What is the total number of employees in the company?")

In [ ]:
# Query 2: Aggregation by categorical feature
ask_database("Give the total number of customers located in each country.")

In [ ]:
# Query 3: Multi-table JOIN (Orders + Customers)
ask_database("What is the name of the customer that placed the highest total order volume, and what is their country?")

In [ ]:
# Query 4: Hierarchical self-join query
ask_database("List all employees along with their job title and the name of the person they report to.")

## 6. Summary & Key Takeaways

1. **Autonomous Tool Selection**: The agent inspects `sql_db_list_tables`, retrieves schemas with `sql_db_schema`, and tests queries before finalizing answers.
2. **Schema Scalability**: Unlike feeding entire tables into LLM context windows, Text-to-SQL agents scale to multi-terabyte production data warehouses.
3. **Security in Production**: Always grant **read-only** database credentials to Text-to-SQL agents and enforce query timeouts.